In [3]:
import os
import pandas as pd

# === 1. Define Base Paths ===
# Go up one level from 'notebooks/' to access the dataset folder
base_path = "../data/raw/BBC News Summary"
articles_path = os.path.join(base_path, "News Articles")
summaries_path = os.path.join(base_path, "Summaries")

# === 2. Safe File Reader Function ===
def read_file_safe(file_path):
    """Reads a text file safely, handling encoding errors automatically."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read().strip()
    except UnicodeDecodeError:
        with open(file_path, 'r', encoding='latin-1') as f:
            return f.read().strip()

# === 3. Combine All Files into One Dataset ===
data = []

for category in os.listdir(articles_path):
    article_dir = os.path.join(articles_path, category)
    summary_dir = os.path.join(summaries_path, category)

    # Skip hidden files like .DS_Store (Mac)
    if category.startswith('.'):
        continue

    for filename in os.listdir(article_dir):
        if not filename.endswith('.txt'):
            continue

        article_file = os.path.join(article_dir, filename)
        summary_file = os.path.join(summary_dir, filename)

        if os.path.exists(summary_file):
            article = read_file_safe(article_file)
            summary = read_file_safe(summary_file)

            data.append({
                'category': category,
                'article': article,
                'summary': summary
            })

# === 4. Create DataFrame ===
df = pd.DataFrame(data)
print(f"✅ Loaded {df.shape[0]} samples from BBC dataset.")
df.head()


✅ Loaded 2225 samples from BBC dataset.


,category,article,summary
0,entertainment,Musicians to tackle US red tape\n\nMusicians' ...,Nigel McCune from the Musicians' Union said Br...
1,entertainment,"U2's desire to be number one\n\nU2, who have w...",But they still want more.They have to want to ...
2,entertainment,Rocker Doherty in on-stage fight\n\nRock singe...,"Babyshambles, which he formed after his acrimo..."
3,entertainment,Snicket tops US box office chart\n\nThe film a...,A Series of Unfortunate Events also stars Scot...
4,entertainment,Ocean's Twelve raids box office\n\nOcean's Twe...,"Ocean's Twelve, the crime caper sequel starrin..."


In [5]:
import re
import nltk
nltk.download('punkt_tab')   # <— new line for newer NLTK versions

from nltk.tokenize import sent_tokenize

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = text.replace('\n', ' ')
    return text.strip()

df['article'] = df['article'].apply(clean_text)
df['summary'] = df['summary'].apply(clean_text)
df['sentences'] = df['article'].apply(sent_tokenize)

print("Text cleaned and tokenized.")
df.head()


[nltk_data] Downloading package punkt_tab to /Users/mac/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Text cleaned and tokenized.


,category,article,summary,sentences
0,entertainment,Musicians to tackle US red tape Musicians' gro...,Nigel McCune from the Musicians' Union said Br...,[Musicians to tackle US red tape Musicians' gr...
1,entertainment,"U2's desire to be number one U2, who have won ...",But they still want more.They have to want to ...,"[U2's desire to be number one U2, who have won..."
2,entertainment,Rocker Doherty in on-stage fight Rock singer P...,"Babyshambles, which he formed after his acrimo...",[Rocker Doherty in on-stage fight Rock singer ...
3,entertainment,Snicket tops US box office chart The film adap...,A Series of Unfortunate Events also stars Scot...,[Snicket tops US box office chart The film ada...
4,entertainment,Ocean's Twelve raids box office Ocean's Twelve...,"Ocean's Twelve, the crime caper sequel starrin...",[Ocean's Twelve raids box office Ocean's Twelv...


In [7]:
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/bbc_dataset.csv", index=False)
print(" Saved cleaned dataset to '../data/processed/bbc_dataset.csv'")


 Saved cleaned dataset to '../data/processed/bbc_dataset.csv'
